# Merged Data Performance Investigation

Investigate whether adding peer university data (Lehigh, Marquette, Villanova)
improves citation prediction performance vs the AUB-only baseline.

**Baseline (AUB-only):**
- F1: 62.55% | ROC-AUC: 81.04% | Recall: 77.15% | Precision: 52.58%
- Train: ~2,605 papers (AUB 2015-2017) | Test: ~3,573 papers (AUB 2018-2020)

**Evaluation scenarios:**
- **Scenario A** — train on all-unis 2015-2017, test on **AUB-only** 2018-2020  
  *Question: does more training data improve AUB predictions?*
- **Scenario B** — train on all-unis 2015-2017, test on **all-unis** 2018-2020  
  *Question: how well does the model generalise across institutions?*

**Inputs:** `data/processed/all_unis_cleaned.pkl` (output of nb 06)

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import pickle
import warnings
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
%matplotlib inline

# ── Baseline reference values ──────────────────────────────────────────────────
BASELINE = {
    'name':      'AUB-only baseline',
    'f1':        0.6255,
    'roc_auc':   0.8104,
    'recall':    0.7715,
    'precision': 0.5258,
    'threshold': 0.54,
    'n_train':   2605,
    'n_test':    3573,
}

## 1. Load Merged Data

In [ ]:
data_path = Path('../data/processed/all_unis_cleaned.pkl')

if not data_path.exists():
    raise FileNotFoundError(
        f"Merged data not found: {data_path}\n"
        "Run notebooks 04 → 05 → 06 first to generate the merged dataset."
    )

df = pd.read_pickle(data_path)

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"\nRows per institution:")
print(df['institution'].value_counts().to_string())
print(f"\nYear range: {df['Year'].min()} – {df['Year'].max()}")
print(f"\nColumns: {df.columns.tolist()}")

## 2. Dataset Overview

In [ ]:
# Year × institution breakdown
year_inst = (
    df.groupby(['Year', 'institution'])
    .size()
    .unstack(fill_value=0)
)
print("Papers per year per institution:")
print(year_inst.to_string())

# Citation distribution
print(f"\nCitation statistics:")
print(df['Citations'].describe().to_string())
print(f"Median: {df['Citations'].median():.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Papers per year stacked by institution
year_inst.plot(kind='bar', stacked=True, ax=axes[0], colormap='Set2')
axes[0].set_title('Papers per Year (all institutions)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Paper count')
axes[0].tick_params(axis='x', rotation=45)

# Citation distribution
axes[1].hist(df['Citations'].clip(upper=100), bins=50, edgecolor='white', color='steelblue')
axes[1].set_title('Citation Distribution (clipped at 100)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Citations')
axes[1].set_ylabel('Papers')

plt.tight_layout()
plt.show()

## 3. Temporal Split

Same years as the AUB-only baseline for a fair comparison:
- **Train**: 2015 – 2017  
- **Test**: 2018 – 2020

In [ ]:
TRAIN_YEARS = [2015, 2016, 2017]
TEST_YEARS  = [2018, 2019, 2020]

train_mask = df['Year'].isin(TRAIN_YEARS)
test_mask  = df['Year'].isin(TEST_YEARS)

df_train = df[train_mask].copy()
df_test  = df[test_mask].copy()

# AUB-only test subset (Scenario A)
df_test_aub = df_test[df_test['institution'] == 'AUB'].copy()

print("SPLIT SUMMARY")
print("=" * 50)
print(f"Train (2015-2017): {len(df_train):,} papers")
print(df_train['institution'].value_counts().to_string())
print(f"\nTest – all unis (2018-2020): {len(df_test):,} papers")
print(df_test['institution'].value_counts().to_string())
print(f"\nTest – AUB-only (2018-2020): {len(df_test_aub):,} papers")
print(f"\nBaseline train: ~{BASELINE['n_train']:,} | test: ~{BASELINE['n_test']:,}")

## 4. Feature Engineering

Same feature set as the AUB-only baseline:
1. **Text features** — TF-IDF on abstracts (5,000 features, bigrams; fit on train only)
2. **Venue features** — SNIP, SJR, CiteScore metrics (if columns present)
3. **Author features** — team size, collaboration indicators

### 4a. Text features (TF-IDF)

In [ ]:
def preprocess_text(text):
    if pd.isna(text):
        return ""
    return str(text).lower()

abstracts_train = df_train['Abstract'].apply(preprocess_text)
abstracts_test_all = df_test['Abstract'].apply(preprocess_text)
abstracts_test_aub = df_test_aub['Abstract'].apply(preprocess_text)

# Fit vectorizer on training data only
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    stop_words='english'
)

tfidf_train = tfidf.fit_transform(abstracts_train)
tfidf_test_all = tfidf.transform(abstracts_test_all)
tfidf_test_aub = tfidf.transform(abstracts_test_aub)

feat_names = [f'tfidf_{f}' for f in tfidf.get_feature_names_out()]

text_train = pd.DataFrame(tfidf_train.toarray(), columns=feat_names, index=df_train.index)
text_test_all = pd.DataFrame(tfidf_test_all.toarray(), columns=feat_names, index=df_test.index)
text_test_aub = pd.DataFrame(tfidf_test_aub.toarray(), columns=feat_names, index=df_test_aub.index)

print(f"TF-IDF train: {text_train.shape}")
print(f"TF-IDF test (all-unis): {text_test_all.shape}")
print(f"TF-IDF test (AUB-only): {text_test_aub.shape}")

### 4b. Venue features (SNIP, SJR, CiteScore)

In [ ]:
def build_venue_features(subset_df):
    """Build venue prestige features (percentile-only).

    Raw scores (snip, citescore, sjr) and venue_score_composite dropped to
    avoid temporal distribution shift between train (2015-17) and test (2018-20).
    See nb21c for rationale (KS up to 0.153 for raw vs ~0.021 for percentiles).
    """
    vf = pd.DataFrame(index=subset_df.index)

    col_map = {
        'snip_percentile':      'SNIP percentile (publication year) *',
        'citescore_percentile': 'CiteScore percentile (publication year) *',
        'sjr_percentile':       'SJR percentile (publication year) *',
    }

    available = {}
    for feat, col in col_map.items():
        if col in subset_df.columns:
            vf[feat] = pd.to_numeric(subset_df[col], errors='coerce')
            available[feat] = col
        else:
            vf[feat] = np.nan

    pct_cols = ['snip_percentile', 'citescore_percentile', 'sjr_percentile']
    vf['avg_venue_percentile'] = vf[pct_cols].mean(axis=1)
    vf['is_top_journal'] = (
        (vf['snip_percentile'] >= 90) |
        (vf['citescore_percentile'] >= 90) |
        (vf['sjr_percentile'] >= 90)
    ).astype(int)

    for col in vf.columns:
        median_val = vf[col].median()
        if pd.isna(median_val):
            median_val = 0
        vf[col] = vf[col].fillna(median_val)

    return vf, available


venue_train, avail_cols = build_venue_features(df_train)
venue_test_all, _ = build_venue_features(df_test)
venue_test_aub, _ = build_venue_features(df_test_aub)

print(f"Venue features: {venue_train.shape[1]} features")
print(f"Available venue columns: {list(avail_cols.keys())}")

### 4c. Author features

In [ ]:
def build_author_features(subset_df):
    """Build author / collaboration features, handling missing columns."""
    af = pd.DataFrame(index=subset_df.index)

    for feat, col in [
        ('num_authors',      'Number of Authors'),
        ('num_institutions', 'Number of Institutions'),
        ('num_countries',    'Number of Countries/Regions'),
    ]:
        af[feat] = pd.to_numeric(subset_df.get(col, pd.Series(np.nan, index=subset_df.index)),
                                 errors='coerce')

    af['is_single_author']     = (af['num_authors'] == 1).astype(int)
    af['is_international_collab'] = (af['num_countries'] > 1).astype(int)
    af['is_multi_institution'] = (af['num_institutions'] > 1).astype(int)
    af['authors_per_institution'] = (
        af['num_authors'] / af['num_institutions'].replace(0, 1)
    )
    af['team_size_small']  = (af['num_authors'] <= 3).astype(int)
    af['team_size_medium'] = ((af['num_authors'] > 3) & (af['num_authors'] <= 10)).astype(int)
    af['team_size_large']  = (af['num_authors'] > 10).astype(int)

    for col in af.columns:
        af[col] = af[col].fillna(af[col].median() if not pd.isna(af[col].median()) else 0)

    return af


author_train    = build_author_features(df_train)
author_test_all = build_author_features(df_test)
author_test_aub = build_author_features(df_test_aub)

print(f"Author features: {author_train.shape[1]} features")

### 4e. Metadata features (Open Access, Topic Prominence, Publication Type, Source Type)

In [ ]:
def build_metadata_features(subset_df, pub_type_cols=None, source_type_cols=None):
    """Build metadata features from nb 22b: open access, topic prominence,
    publication type (one-hot), source type (one-hot).

    pub_type_cols / source_type_cols: column list from training set, used to
    align test sets to the same dummy columns.
    """
    mf = pd.DataFrame(index=subset_df.index)

    # Open Access
    mf['is_open_access'] = subset_df.get(
        'Open Access', pd.Series(np.nan, index=subset_df.index)
    ).notna().astype(int)

    # Topic Prominence
    mf['topic_prominence'] = pd.to_numeric(
        subset_df.get('Topic Prominence Percentile',
                      pd.Series(np.nan, index=subset_df.index)),
        errors='coerce'
    )
    median_tp = mf['topic_prominence'].median()
    mf['topic_prominence'] = mf['topic_prominence'].fillna(0 if pd.isna(median_tp) else median_tp)

    # Publication type (one-hot)
    pub_dummies = pd.get_dummies(
        subset_df.get('Publication type', pd.Series(dtype=str)),
        prefix='pubtype', dummy_na=False
    )
    if pub_type_cols is not None:
        pub_dummies = pub_dummies.reindex(columns=pub_type_cols, fill_value=0)

    # Source type (one-hot)
    src_dummies = pd.get_dummies(
        subset_df.get('Source type', pd.Series(dtype=str)),
        prefix='sourcetype', dummy_na=False
    )
    if source_type_cols is not None:
        src_dummies = src_dummies.reindex(columns=source_type_cols, fill_value=0)

    mf = pd.concat([mf, pub_dummies, src_dummies], axis=1)
    return mf, list(pub_dummies.columns), list(src_dummies.columns)


meta_train, pub_type_cols, source_type_cols = build_metadata_features(df_train)
meta_test_all, _, _ = build_metadata_features(df_test,    pub_type_cols, source_type_cols)
meta_test_aub, _, _ = build_metadata_features(df_test_aub, pub_type_cols, source_type_cols)

print(f"Metadata features: {meta_train.shape[1]} features")
print(f"  is_open_access:    1")
print(f"  topic_prominence:  1")
print(f"  pubtype dummies:   {len(pub_type_cols)}")
print(f"  sourcetype dummies:{len(source_type_cols)}")

### 4d. Combine feature matrices

In [ ]:
X_train     = pd.concat([text_train, venue_train, author_train, meta_train], axis=1)
X_test_all  = pd.concat([text_test_all, venue_test_all, author_test_all, meta_test_all], axis=1)
X_test_aub  = pd.concat([text_test_aub, venue_test_aub, author_test_aub, meta_test_aub], axis=1)

print(f"Feature matrix – train:       {X_train.shape}")
print(f"Feature matrix – test (all):  {X_test_all.shape}")
print(f"Feature matrix – test (AUB):  {X_test_aub.shape}")
print(f"\nFeature breakdown:")
print(f"  TF-IDF:   {text_train.shape[1]}")
print(f"  Venue:    {venue_train.shape[1]}")
print(f"  Author:   {author_train.shape[1]}")
print(f"  Metadata: {meta_train.shape[1]}")
print(f"  Total:    {X_train.shape[1]}")

## 5. Build Targets

In [ ]:
# Define high-impact threshold on TRAINING data (top 25%)
citation_threshold = df_train['Citations'].quantile(0.75)

y_train     = (df_train['Citations'] >= citation_threshold).astype(int)
y_test_all  = (df_test['Citations']  >= citation_threshold).astype(int)
y_test_aub  = (df_test_aub['Citations'] >= citation_threshold).astype(int)

print(f"Citation threshold (top 25% of train): {citation_threshold:.0f}")
print(f"\nTrain high-impact:    {y_train.sum():,} / {len(y_train):,} ({y_train.mean()*100:.1f}%)")
print(f"Test (all) high-impact: {y_test_all.sum():,} / {len(y_test_all):,} ({y_test_all.mean()*100:.1f}%)")
print(f"Test (AUB) high-impact: {y_test_aub.sum():,} / {len(y_test_aub):,} ({y_test_aub.mean()*100:.1f}%)")

## 6. Train Logistic Regression (same settings as baseline)

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

print("Training LogisticRegression on merged training set...")
model.fit(X_train, y_train)
print("Done.")

# Probability predictions
proba_test_all = model.predict_proba(X_test_all)[:, 1]
proba_test_aub = model.predict_proba(X_test_aub)[:, 1]

print(f"\nProbability range (all-unis test): [{proba_test_all.min():.3f}, {proba_test_all.max():.3f}]")
print(f"Probability range (AUB test):      [{proba_test_aub.min():.3f}, {proba_test_aub.max():.3f}]")

## 7. Threshold Optimisation

In [ ]:
def find_optimal_threshold(y_true, y_proba, thresholds=None):
    """Return the threshold that maximises F1 score."""
    if thresholds is None:
        thresholds = np.arange(0.10, 0.90, 0.01)
    f1s = [f1_score(y_true, (y_proba >= t).astype(int)) for t in thresholds]
    best_idx = int(np.argmax(f1s))
    return thresholds[best_idx], f1s[best_idx], f1s


thresholds = np.arange(0.10, 0.90, 0.01)

opt_t_all, opt_f1_all, f1s_all = find_optimal_threshold(y_test_all, proba_test_all, thresholds)
opt_t_aub, opt_f1_aub, f1s_aub = find_optimal_threshold(y_test_aub, proba_test_aub, thresholds)

print(f"Optimal threshold (all-unis test): {opt_t_all:.2f}  →  F1 = {opt_f1_all*100:.2f}%")
print(f"Optimal threshold (AUB-only test): {opt_t_aub:.2f}  →  F1 = {opt_f1_aub*100:.2f}%")
print(f"Baseline threshold: {BASELINE['threshold']:.2f}  →  F1 = {BASELINE['f1']*100:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, f1s, opt_t, opt_f1, label in [
    (axes[0], f1s_all, opt_t_all, opt_f1_all, 'All-unis test'),
    (axes[1], f1s_aub, opt_t_aub, opt_f1_aub, 'AUB-only test'),
]:
    ax.plot(thresholds, f1s, color='steelblue', linewidth=2, label='F1 (merged model)')
    ax.axvline(opt_t, color='red', linestyle='--', linewidth=1.5,
               label=f'Optimal ({opt_t:.2f}) → {opt_f1*100:.2f}%')
    ax.axhline(BASELINE['f1'], color='orange', linestyle=':', linewidth=1.5,
               label=f'AUB baseline ({BASELINE["f1"]*100:.2f}%)')
    ax.set_title(f'Threshold vs F1 — {label}', fontweight='bold')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('F1 Score')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
figures_dir = Path('../reports/figures')
figures_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(figures_dir / 'merged_data_threshold_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Full Metrics at Optimal Threshold

In [ ]:
def compute_metrics(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        'f1':        f1_score(y_true, y_pred),
        'roc_auc':   roc_auc_score(y_true, y_proba),
        'recall':    recall_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'threshold': threshold,
        'n_test':    len(y_true),
        'n_train':   len(y_train),
    }


metrics_all = compute_metrics(y_test_all, proba_test_all, opt_t_all)
metrics_aub = compute_metrics(y_test_aub, proba_test_aub, opt_t_aub)

# Build comparison table
results = pd.DataFrame({
    'AUB-only baseline':              BASELINE,
    'Merged model (AUB-only test)':   metrics_aub,
    'Merged model (all-unis test)':   metrics_all,
}).T

display_cols = ['f1', 'roc_auc', 'recall', 'precision', 'threshold', 'n_train', 'n_test']
results_display = results[display_cols].copy()

for col in ['f1', 'roc_auc', 'recall', 'precision']:
    results_display[col] = results_display[col].apply(lambda x: f"{float(x)*100:.2f}%")

results_display['threshold'] = results_display['threshold'].apply(lambda x: f"{float(x):.2f}")
results_display[['n_train', 'n_test']] = results_display[['n_train', 'n_test']].apply(
    lambda c: c.apply(lambda x: f"{int(float(x)):,}")
)

print(results_display.to_string())

## 9. Scenario A — AUB-only Test: Δ vs Baseline

In [ ]:
delta_f1  = (metrics_aub['f1']      - BASELINE['f1'])      * 100
delta_auc = (metrics_aub['roc_auc'] - BASELINE['roc_auc']) * 100
delta_rec = (metrics_aub['recall']  - BASELINE['recall'])  * 100
delta_pre = (metrics_aub['precision'] - BASELINE['precision']) * 100

print("SCENARIO A: More training data (merged) → same AUB test set")
print("=" * 60)
print(f"{'Metric':<20} {'Baseline':>12} {'Merged':>12} {'Δ':>10}")
print("-" * 60)
print(f"{'F1':<20} {BASELINE['f1']*100:>11.2f}% {metrics_aub['f1']*100:>11.2f}% {delta_f1:>+9.2f}pp")
print(f"{'ROC-AUC':<20} {BASELINE['roc_auc']*100:>11.2f}% {metrics_aub['roc_auc']*100:>11.2f}% {delta_auc:>+9.2f}pp")
print(f"{'Recall':<20} {BASELINE['recall']*100:>11.2f}% {metrics_aub['recall']*100:>11.2f}% {delta_rec:>+9.2f}pp")
print(f"{'Precision':<20} {BASELINE['precision']*100:>11.2f}% {metrics_aub['precision']*100:>11.2f}% {delta_pre:>+9.2f}pp")
print("-" * 60)
print(f"Train size:  {BASELINE['n_train']:,} → {len(y_train):,} papers (+{len(y_train)-BASELINE['n_train']:,})")

if delta_f1 > 0:
    print(f"\n✅ F1 IMPROVED by {delta_f1:+.2f} percentage points")
elif delta_f1 < -0.5:
    print(f"\n❌ F1 DEGRADED by {abs(delta_f1):.2f} percentage points (peer data may introduce noise)")
else:
    print(f"\n➡️  F1 UNCHANGED (Δ={delta_f1:+.2f}pp) — peer data neither helps nor hurts AUB predictions")

## 10. Scenario B — Cross-institution Generalisation

In [ ]:
# Per-institution metrics on the all-unis test set
inst_metrics = {}
for inst in sorted(df_test['institution'].unique()):
    mask = df_test['institution'] == inst
    if mask.sum() < 20:
        continue
    y_i = y_test_all[mask]
    p_i = proba_test_all[mask]
    inst_metrics[inst] = {
        'n_papers':  int(mask.sum()),
        'f1':        f1_score(y_i, (p_i >= opt_t_all).astype(int)),
        'roc_auc':   roc_auc_score(y_i, p_i),
        'recall':    recall_score(y_i, (p_i >= opt_t_all).astype(int)),
        'precision': precision_score(y_i, (p_i >= opt_t_all).astype(int)),
    }

inst_df = pd.DataFrame(inst_metrics).T
print("Per-institution metrics (test 2018-2020):")
for col in ['f1', 'roc_auc', 'recall', 'precision']:
    inst_df[f'{col}_pct'] = inst_df[col].apply(lambda x: f"{x*100:.2f}%")

display_inst = inst_df[['n_papers', 'f1_pct', 'roc_auc_pct', 'recall_pct', 'precision_pct']]
display_inst.columns = ['n_papers', 'F1', 'ROC-AUC', 'Recall', 'Precision']
print(display_inst.to_string())

## 11. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_true, y_proba, threshold, title in [
    (axes[0], y_test_aub, proba_test_aub, opt_t_aub,
     f'Merged model — AUB-only test\n(threshold={opt_t_aub:.2f}, F1={metrics_aub["f1"]*100:.2f}%)'),
    (axes[1], y_test_all, proba_test_all, opt_t_all,
     f'Merged model — all-unis test\n(threshold={opt_t_all:.2f}, F1={metrics_all["f1"]*100:.2f}%)'),
]:
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Low', 'High'], yticklabels=['Low', 'High'])
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig(figures_dir / 'merged_data_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Visual Summary — Merged vs Baseline

In [ ]:
metrics_to_plot = ['f1', 'roc_auc', 'recall', 'precision']
labels = ['F1', 'ROC-AUC', 'Recall', 'Precision']

x = np.arange(len(metrics_to_plot))
width = 0.25

vals_baseline = [BASELINE[m] for m in metrics_to_plot]
vals_aub      = [metrics_aub[m] for m in metrics_to_plot]
vals_all      = [metrics_all[m] for m in metrics_to_plot]

fig, ax = plt.subplots(figsize=(10, 6))

b1 = ax.bar(x - width, vals_baseline, width, label='AUB-only baseline', color='steelblue', alpha=0.8)
b2 = ax.bar(x,         vals_aub,      width, label='Merged model (AUB-only test)', color='darkorange', alpha=0.8)
b3 = ax.bar(x + width, vals_all,      width, label='Merged model (all-unis test)', color='forestgreen', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.05)
ax.set_title('Merged Data vs AUB-only Baseline', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, axis='y', alpha=0.3)

# Value labels
for bars in [b1, b2, b3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height*100:.1f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(figures_dir / 'merged_vs_baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Summary & Conclusions

In [ ]:
print("=" * 65)
print("MERGED DATA PERFORMANCE INVESTIGATION — SUMMARY")
print("=" * 65)

print(f"\n{'Model / Test Set':<35} {'F1':>8} {'ROC-AUC':>9} {'Recall':>8} {'Precision':>10}")
print("-" * 75)
print(f"{'AUB-only baseline':<35} {BASELINE['f1']*100:>7.2f}% {BASELINE['roc_auc']*100:>8.2f}% "
      f"{BASELINE['recall']*100:>7.2f}% {BASELINE['precision']*100:>9.2f}%")
print(f"{'Merged (AUB-only test) — Scenario A':<35} {metrics_aub['f1']*100:>7.2f}% "
      f"{metrics_aub['roc_auc']*100:>8.2f}% {metrics_aub['recall']*100:>7.2f}% "
      f"{metrics_aub['precision']*100:>9.2f}%")
print(f"{'Merged (all-unis test) — Scenario B':<35} {metrics_all['f1']*100:>7.2f}% "
      f"{metrics_all['roc_auc']*100:>8.2f}% {metrics_all['recall']*100:>7.2f}% "
      f"{metrics_all['precision']*100:>9.2f}%")
print("-" * 75)

print(f"\nSCENARIO A — Impact of more training data on AUB predictions:")
print(f"  F1 change:      {delta_f1:+.2f} percentage points")
print(f"  ROC-AUC change: {delta_auc:+.2f} percentage points")
print(f"  Train set grew: {BASELINE['n_train']:,} → {len(y_train):,} papers")

print(f"\nSCENARIO B — Cross-institution generalisation:")
print(f"  Overall F1 (all-unis test): {metrics_all['f1']*100:.2f}%")
print(f"  Per-institution breakdown:")
for inst, m in inst_metrics.items():
    print(f"    {inst:<12}: F1={m['f1']*100:.2f}%  ROC-AUC={m['roc_auc']*100:.2f}%  (n={m['n_papers']:,})")

print(f"\nCONCLUSION:")
if delta_f1 > 1.0:
    print("  Merging peer university data meaningfully improves F1 on AUB papers.")
    print("  More training data from similar institutions helps the model generalise.")
elif delta_f1 > 0:
    print("  Merging peer university data marginally improves AUB F1.")
    print("  Effect is modest — dataset size may still be the primary bottleneck.")
elif delta_f1 > -0.5:
    print("  Merging peer university data leaves AUB F1 essentially unchanged.")
    print("  Peer data neither helps nor hurts AUB-specific predictions.")
else:
    print("  Merging peer university data degrades AUB F1.")
    print("  Possible reason: citation distributions or field mixes differ across institutions.")
    print("  Consider training institution-specific models or reweighting peer data.")

## 14. Save Artifacts

In [ ]:
# Save model
models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)

with open(models_dir / 'merged_data_logistic_regression.pkl', 'wb') as f:
    pickle.dump(model, f)

with open(models_dir / 'merged_data_tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Save results CSV
metrics_dir = Path('../reports/metrics')
metrics_dir.mkdir(parents=True, exist_ok=True)

results_raw = pd.DataFrame({
    'AUB-only baseline': BASELINE,
    'Merged model (AUB-only test)': metrics_aub,
    'Merged model (all-unis test)': metrics_all,
}).T
results_raw.to_csv(metrics_dir / 'merged_data_performance.csv')

if inst_metrics:
    pd.DataFrame(inst_metrics).T.to_csv(metrics_dir / 'merged_data_per_institution.csv')

print("Saved:")
print(f"  models/merged_data_logistic_regression.pkl")
print(f"  models/merged_data_tfidf_vectorizer.pkl")
print(f"  reports/metrics/merged_data_performance.csv")
print(f"  reports/metrics/merged_data_per_institution.csv")
print(f"  reports/figures/merged_data_threshold_optimization.png")
print(f"  reports/figures/merged_data_confusion_matrices.png")
print(f"  reports/figures/merged_vs_baseline_comparison.png")